# Fortgeschrittene Empfehlungssysteme

In [1]:
import numpy as np
import pandas as pd

## Ratings laden

In [2]:
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv('U.data', sep='\t', names=column_names)

Daten ansehen

In [3]:
df.head()

,user_id,item_id,rating,timestamp
0,0,50,5,881250949
1,0,172,5,881250949
2,0,133,1,881250949
3,196,242,3,881250949
4,186,302,3,891717742


Filmtitel ergänzen

In [4]:
movie_titles = pd.read_csv("U.item", sep="|", header=None, usecols=[0,1], names=["item_id", "title"], encoding="latin-1")
movie_titles.head()

Merge

In [ ]:
df = pd.merge(df,movie_titles,on='item_id')
df.head()

Anzahl Nutzer und Filme

In [ ]:
n_users = df.user_id.nunique()
n_items = df.item_id.nunique()

print('Anzahl an Nutzern: '+ str(n_users))
print('Anzahl an Filmen: '+str(n_items))

## Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(df, test_size=0.25)

## Memory-Based Collaborative Filtering

In [ ]:
train_data_matrix = np.zeros((n_users, n_items))
for line in train_data.itertuples():
    train_data_matrix[line[1]-1, line[2]-1] = line[3]  

test_data_matrix = np.zeros((n_users, n_items))
for line in test_data.itertuples():
    test_data_matrix[line[1]-1, line[2]-1] = line[3]

Ähnlichkeitsmatrizen

In [ ]:
from sklearn.metrics.pairwise import pairwise_distances
user_similarity = pairwise_distances(train_data_matrix, metric='cosine')
item_similarity = pairwise_distances(train_data_matrix.T, metric='cosine')

Vorhersagen

In [ ]:
def predict(ratings, similarity, type='user'):
    if type == 'user':
        mean_user_rating = ratings.mean(axis=1)
        ratings_diff = ratings - mean_user_rating[:, np.newaxis]
        pred = mean_user_rating[:, np.newaxis] + similarity.dot(ratings_diff) / np.array([np.abs(similarity).sum(axis=1)]).T
    elif type == 'item':
        pred = ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)])
    return pred


In [ ]:
item_prediction = predict(train_data_matrix, item_similarity, type='item')
user_prediction = predict(train_data_matrix, user_similarity, type='user')

## Auswertung

RMSE

In [ ]:
from sklearn.metrics import mean_squared_error
from math import sqrt
def rmse(prediction, ground_truth):
    prediction = prediction[ground_truth.nonzero()].flatten() 
    ground_truth = ground_truth[ground_truth.nonzero()].flatten()
    return sqrt(mean_squared_error(prediction, ground_truth))

In [ ]:
print('User-based CF RMSE: ' + str(rmse(user_prediction, test_data_matrix)))
print('Item-based CF RMSE: ' + str(rmse(item_prediction, test_data_matrix)))

## Hinweis

Memory-based CF

In [ ]:
sparsity=round(1.0-len(df)/float(n_users*n_items),3)
print('Das Seltenheits-Niveau ist ' +  str(sparsity*100) + '%')

## Matrixfaktorisierung

In [ ]:
import scipy.sparse as sp
from scipy.sparse.linalg import svds

u, s, vt = svds(train_data_matrix, k=20)
s_diag_matrix = np.diag(s)
X_pred = np.dot(np.dot(u, s_diag_matrix), vt)
print('User-based CF MSE: ' + str(rmse(X_pred, test_data_matrix)))


## Hinweis

SVD und Overfitting